In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, vonmises_fisher
import seaborn as sns
import sys
import glob
from tqdm import trange 

In [ ]:
results = "../../results/figure5/*.csv"

In [4]:
dfs = list()
for f in glob.glob(results):
    dfs.append(pd.read_csv(f).rename({"Unnamed: 0": "measure"}, axis=1))

In [5]:
all_dfs = pd.concat(dfs)
all_dfs = all_dfs[all_dfs["measure"].isin(["betweenness", "degree", "closeness", "clustering", "pagerank"])].copy()

In [6]:
all_dfs.columns = ['measure', 'raw', 'BOSPERRUS', 'SERN', 'on crop vs. SERN values', 'cap_radius', 'graph_type', 'coord_type', 'n', 'k', 'radius']

In [7]:
all_dfs["radius"] = all_dfs["radius"].fillna(0)
all_dfs["k"] = all_dfs["k"].fillna(0)
all_dfs["param"] = all_dfs["k"] + all_dfs["radius"]

In [8]:
all_dfs["BOSPERRUS"] = np.abs(all_dfs["BOSPERRUS"])
all_dfs["SERN"] = np.abs(all_dfs["SERN"])
all_dfs["raw"] = np.abs(all_dfs["raw"])

In [9]:
all_dfs = all_dfs.reset_index()

In [10]:
all_dfs = all_dfs.iloc[all_dfs[["BOSPERRUS", "SERN", "raw"]].dropna(how="all").index]

In [11]:
best_corr = all_dfs[["BOSPERRUS", "SERN", "raw"]].idxmax(axis=1)
all_dfs["highest correlation"] = best_corr

In [ ]:
from scipy.stats import mannwhitneyu

def _get_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def _draw_bracket(ax, x1, x2, y, stars, fontsize=7):
    h = 0.025
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y],
            lw=0.8, color="k", clip_on=False)
    ax.text((x1 + x2) / 2, y + h, stars,
            ha="center", va="bottom", fontsize=fontsize, clip_on=False)

df_long = all_dfs.melt(
    id_vars=["measure", "graph_type", "coord_type"],
    value_vars=["raw", "BOSPERRUS", "SERN"],
    var_name="method",
    value_name="correlation",
).dropna(subset=["correlation"])

METHOD_PALETTE = {"raw": "#7DE3A1", "BOSPERRUS": "#7DE3D4", "SERN": "#7DBFE3"}
METHOD_ORDER = ["raw", "BOSPERRUS", "SERN"]
ROW_ORDER = ["delaunay", "knn", "rnn"]
MEASURE_ORDER = sorted(df_long["measure"].unique())
ROW_LABELS = {"delaunay": "Delaunay", "knn": "$k$-NN", "rnn": "$r$-NN"}

g = sns.catplot(
    data=df_long,
    x="method", y="correlation", hue="method",
    col="measure", row="graph_type",
    kind="violin",
    order=METHOD_ORDER,
    row_order=ROW_ORDER,
    col_order=MEASURE_ORDER,
    palette=METHOD_PALETTE,
    inner="box",
    cut=0,
    linewidth=0.8,
    height=2.5, aspect=0.9,
    sharey=True,
    legend=False,
)

g.set_titles("")
for ax, col in zip(g.axes[0], MEASURE_ORDER):
    ax.set_title(col.capitalize(), fontsize=10)
for ax in g.axes.flat:
    ax.set_xlabel("")
    ax.set_ylabel("")
for ax_row, row_key in zip(g.axes[:, 0], ROW_ORDER):
    ax_row.set_ylabel(f"{ROW_LABELS[row_key]}\nPearson $r$")
for ax in g.axes[-1]:
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)

# Paired lines + MWU significance brackets per facet
N_SAMPLE = 200
rng = np.random.default_rng(42)
SIG_PAIRS = [
    ("raw", "BOSPERRUS", 0, 1, 1.04),   # adjacent, lowest bracket
    ("BOSPERRUS", "SERN",  1, 2, 1.11),  # adjacent
    ("raw", "SERN",        0, 2, 1.19),  # spanning, highest bracket
]

for row_idx, row_key in enumerate(ROW_ORDER):
    for col_idx, col_key in enumerate(MEASURE_ORDER):
        ax = g.axes[row_idx][col_idx]
        subset = all_dfs[
            (all_dfs["graph_type"] == row_key) &
            (all_dfs["measure"] == col_key)
        ][["raw", "BOSPERRUS", "SERN"]]

        # --- paired lines (sampled subset for readability) ---
        s = subset.dropna(how="all")
        idx = rng.choice(len(s), size=min(N_SAMPLE, len(s)), replace=False)
        sample = s.iloc[idx]

        xs_all, ys_all = [], []
        for _, row_data in sample.iterrows():
            pts = [(i, v) for i, v in enumerate(
                [row_data["raw"], row_data["BOSPERRUS"], row_data["SERN"]]
            ) if not np.isnan(v)]
            if len(pts) >= 2:
                xs_all.extend([p[0] for p in pts] + [np.nan])
                ys_all.extend([p[1] for p in pts] + [np.nan])
        if xs_all:
            ax.plot(xs_all, ys_all, color="k", alpha=0.05, lw=0.4,
                    zorder=0, solid_capstyle="round")

        # --- MWU significance brackets ---
        for a, b, x1, x2, y in SIG_PAIRS:
            va = subset[a].dropna()
            vb = subset[b].dropna()
            if len(va) > 1 and len(vb) > 1:
                _, p = mannwhitneyu(va, vb, alternative="two-sided")
                _draw_bracket(ax, x1, x2, y, _get_stars(p))

g.set(ylim=(-0.05, 1.33))
g.figure.tight_layout()
plt.savefig("../../result_plots/figure5/correlations_violin.pdf", bbox_inches="tight")